# More Pandas Practice: Combining Data, Binning, and Pivot Tables

## Starter Code

In [ ]:
# Import pandas and load both wine datasets, run this cell first
import pandas as pd

red_df = pd.read_csv("../../data/winequality-red.csv", delimiter=";")
white_df = pd.read_csv("../../data/winequality-white.csv", delimiter=";")

print("Red wine shape:", red_df.shape)
print("White wine shape:", white_df.shape)

---

## Part 1: Loading and Inspecting

### Challenge 1: Sense-check the data

Use DataFrame attributes and methods to confirm both datasets loaded correctly. Check:

- The number of rows and columns in each

- The data types of the columns

- Whether any columns contain null values

In [ ]:
# Number of rows and columns in each dataset
print("Red shape:", red_df.shape)
print("White shape:", white_df.shape)

# Column data types
print("\nRed dtypes:")
print(red_df.dtypes)
print("\nWhite dtypes:")
print(white_df.dtypes)

# Count null values; a total of 0 means there are no missing entries
print("\nRed nulls:", red_df.isnull().sum().sum())
print("White nulls:", white_df.isnull().sum().sum())

**Explanation**

Confirms both datasets loaded cleanly: .shape gives rows and columns, .dtypes shows each column is numeric, and .isnull().sum().sum() totals missing values (0 means none). A quick sense-check like this catches load problems such as the wrong delimiter before you build on the data.


---

## Part 2: Filtering by Alcohol Content

### Challenge 2: Flag above-average alcohol

You want to avoid wines with above-average alcohol content. To do this:

1. Find the **mean alcohol content** separately for reds and whites.

2. Create a **boolean `Series`** for each DataFrame indicating whether each row has an alcohol content above the mean.

3. Attach this boolean `Series` to each DataFrame, do it **twice**: once using `.join()` and once using `pd.concat(..., axis=1)`.

4. Filter each DataFrame to return only the rows where alcohol is **above** the mean.

In [ ]:
# Step 1: Calculate mean alcohol for each wine type
red_mean_alcohol = red_df["alcohol"].mean()
white_mean_alcohol = white_df["alcohol"].mean()

print("Red mean alcohol:", red_mean_alcohol)
print("White mean alcohol:", white_mean_alcohol)

**Explanation**

Series.mean() computes the average alcohol for each wine type separately. Keeping reds and whites apart matters because their alcohol distributions differ, so a single shared mean would mislabel rows.


In [ ]:
# Step 2: Create boolean Series, True where alcohol is above the mean
# name the Series so it becomes a tidy column name once attached
red_above_mean = (red_df["alcohol"] > red_mean_alcohol).rename("above_mean_alcohol")
white_above_mean = (white_df["alcohol"] > white_mean_alcohol).rename(
    "above_mean_alcohol"
)

print(red_above_mean.head())

**Explanation**

Comparing the alcohol column to its mean returns a boolean Series, True where a row is above average. We .rename() it so that once attached it lands in a tidy, named column instead of an unnamed one.


In [ ]:
# Step 3a: Attach using .join()
# .join() aligns on the index, which both objects share here
red_joined = red_df.join(red_above_mean)
white_joined = white_df.join(white_above_mean)

print(red_joined.head())

**Explanation**

.join() attaches the boolean Series to the DataFrame by matching on the shared index. It is concise for index-aligned data; if the indexes did not line up you would get NaN where they failed to match.


In [ ]:
# Step 3b: Attach using pd.concat(..., axis=1)
# axis=1 glues columns side by side, also aligning on the index
red_concat = pd.concat([red_df, red_above_mean], axis=1)
white_concat = pd.concat([white_df, white_above_mean], axis=1)

print(red_concat.head())

**Explanation**

pd.concat(..., axis=1) produces the same result by gluing columns side by side, also aligning on the index. axis=1 means combine columns (axis=0 would stack rows instead), so .join() and concat are interchangeable here.


In [ ]:
# Step 4: Filter to only rows where alcohol is above the mean
red_above = red_joined[red_joined["above_mean_alcohol"]]
white_above = white_joined[white_joined["above_mean_alcohol"]]

print("Red above mean rows:", red_above.shape[0])
print("White above mean rows:", white_above.shape[0])

**Explanation**

Passing the boolean column inside square brackets keeps only the rows where the flag is True, the above-average alcohol wines. This is standard boolean masking, the same pattern used for any conditional row filter.


---

## Part 3: Binning by Acidity

### Challenge 3: Bin by fixed acidity

You want to avoid wines with the highest acidity. To do this:

1. Use `pd.cut()` to divide the `fixed acidity` column of each DataFrame into **5 equal-width bins**.

2. Attach the resulting bin `Series` to each original DataFrame, once using `.join()` and once using `pd.concat(..., axis=1)`.

3. Filter each DataFrame to return only rows that are **not** in the top acidity bin.

> Note: equal-width bins divide the *range* of values into equal intervals. They are different from quintiles (equal-count bins).

In [ ]:
# Step 1: Create 5 equal-width acidity bins for red and white wines
# pd.cut with an integer splits the value range into that many equal-width intervals
red_acidity_bin = pd.cut(red_df["fixed acidity"], bins=5).rename("acidity_bin")
white_acidity_bin = pd.cut(white_df["fixed acidity"], bins=5).rename("acidity_bin")

print(red_acidity_bin.value_counts().sort_index())

**Explanation**

pd.cut with bins=5 splits the fixed acidity range into 5 equal-width intervals and labels each row with its interval. Equal-width bins divide the value range evenly, so some bins can hold far more rows than others, which is different from quantile bins of equal count.


In [ ]:
# Step 2a: Attach bins using .join()
red_binned = red_df.join(red_acidity_bin)
white_binned = white_df.join(white_acidity_bin)

print(red_binned.head())

**Explanation**

.join() attaches the named bin Series to the DataFrame on the shared index, adding an acidity_bin column. Same index-alignment idea as Challenge 2.


In [ ]:
# Step 2b: Attach bins using pd.concat(..., axis=1)
red_binned_concat = pd.concat([red_df, red_acidity_bin], axis=1)
white_binned_concat = pd.concat([white_df, white_acidity_bin], axis=1)

print(red_binned_concat.head())

**Explanation**

pd.concat(..., axis=1) attaches the same bin column by aligning on the index, the concat equivalent of the .join() above.


In [ ]:
# Step 3: Filter out the top acidity bin from each DataFrame
# the categories are ordered low to high, so the last one is the highest acidity
red_top_bin = red_binned["acidity_bin"].cat.categories[-1]
white_top_bin = white_binned["acidity_bin"].cat.categories[-1]

red_not_top = red_binned[red_binned["acidity_bin"] != red_top_bin]
white_not_top = white_binned[white_binned["acidity_bin"] != white_top_bin]

print("Red rows kept:", red_not_top.shape[0])
print("White rows kept:", white_not_top.shape[0])

**Explanation**

The bin categories are ordered low to high, so categories[-1] is the highest-acidity interval; keeping rows where acidity_bin is not that category drops the top bin. Using the category label rather than a hard-coded interval keeps the code correct even if the data range changes.


---

## Part 4: Pivot Tables

### Challenge 4: Alcohol by quality, pivot table

You want to understand how avoiding above-average alcohol affects the quality of wines available to you.

Using the **above-average alcohol** subsets from Challenge 2, create a **pivot table** for each wine type showing:

- The **average `alcohol` content** as the value

- **`quality`** as the index

This should give you one row per quality level and the average alcohol content for that group.

In [ ]:
# Pivot table for red wines above mean alcohol, average alcohol by quality
# mean is the default aggregation, shown explicitly here for clarity
red_pivot = pd.pivot_table(red_above, values="alcohol", index="quality", aggfunc="mean")
red_pivot

**Explanation**

pd.pivot_table with index='quality' and the alcohol values gives one row per quality score with the group mean. mean is the default aggfunc but is written out so the intent is obvious.


In [ ]:
# Pivot table for white wines above mean alcohol, average alcohol by quality
white_pivot = pd.pivot_table(
    white_above, values="alcohol", index="quality", aggfunc="mean"
)
white_pivot

**Explanation**

Same pivot as the red case applied to the white above-mean subset, letting you compare how average alcohol varies by quality for whites.


### Challenge 5: Alcohol by quality and acidity bin, pivot table

Now do the same for the **non-top-acidity** subsets from Challenge 3. Create a pivot table for each wine type showing:

- The **average `alcohol` content** as the value

- **`quality`** as the index

- **`fixed acidity` bin** as the columns

This lets you see average alcohol broken down by both quality score and acidity category.

In [ ]:
# Pivot table for red wines, average alcohol by quality and acidity bin
red_pivot_2d = pd.pivot_table(
    red_not_top,
    values="alcohol",
    index="quality",
    columns="acidity_bin",
    aggfunc="mean",
    observed=False,  # keep all bin categories as columns even if some are empty
)
red_pivot_2d

**Explanation**

Adding columns='acidity_bin' turns the pivot into a 2D grid of average alcohol by quality (rows) and acidity bin (columns). observed=False keeps every bin category as a column even when a quality and bin combination has no rows, so the grid stays complete; empty cells show as NaN.


In [ ]:
# Pivot table for white wines, average alcohol by quality and acidity bin
white_pivot_2d = pd.pivot_table(
    white_not_top,
    values="alcohol",
    index="quality",
    columns="acidity_bin",
    aggfunc="mean",
    observed=False,  # keep all bin categories as columns even if some are empty
)
white_pivot_2d

**Explanation**

Same 2D pivot applied to the white non-top-acidity subset, giving the quality-by-acidity breakdown of average alcohol for whites.


---

## Part 5: Open Exploration

### Challenge 6: Your own question

Set a 10-minute timer. Pick one question about the wine data that interests you and answer it using any combination of the tools from this notebook (filtering, joining, binning, pivot tables, groupby, visualisation). Be ready to explain what you found and why you chose that question.

In [ ]:
# Example question: does higher quality go with higher average alcohol in each wine type?
# (this is one possible answer; any sensible question using the notebook tools is fine)
red_quality_alcohol = pd.pivot_table(
    red_df, values="alcohol", index="quality", aggfunc="mean"
)
white_quality_alcohol = pd.pivot_table(
    white_df, values="alcohol", index="quality", aggfunc="mean"
)

print("Red: average alcohol by quality")
print(red_quality_alcohol)
print("\nWhite: average alcohol by quality")
print(white_quality_alcohol)

# Takeaway: for both reds and whites, the highest quality scores tend to have the
# highest average alcohol, suggesting alcohol content and rated quality move together.